[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    B, S, D = q.shape
    B, S, D = q.shape
    pos = torch.arange(S, device=q.device).float()
    idx = torch.arange(D//2, device=q.device).float()
    freqs = 1.0 / 10000 ** (2 * idx / D)
    theta = pos[:, None] * freqs[None, :] # [S, D]
    # print(theta.shape, B, S, D)

    sin = torch.sin(theta)
    cos = torch.cos(theta)

    def apply_rope(x, sin, cos):
        B, S, D = x.shape
        x_0 = x[:, :, :D//2]
        x_1 = x[:, :, D//2:]
        y0 = x_0 * cos[None, :, :] - x_1 * sin[None, :, :]
        y1 = x_0 * sin[None, :, :] + x_1 * cos[None, :, :]
        y = torch.concat([y0, y1], dim=-1)
        return y

    roped_q = apply_rope(q, sin, cos)
    roped_k = apply_rope(k, sin, cos)
    return roped_q, roped_k


In [28]:
def apply_rope(q, k):
    B, S, D = q.shape
    B, S_k, D = k.shape

    S = max(S, S_k)

    pos = torch.arange(S, device=q.device).float()
    idx = torch.arange(D//2, device=q.device).float()
    freqs = 1.0 / 10000 ** (2 * idx / D)
    theta = pos[:, None] * freqs[None, :] # [S, D//2]
    # print(f"{theta.shape} {theta.max()=} {theta.min()=}")

    sin = torch.sin(theta)
    cos = torch.cos(theta)

    def apply_rope_(x, sin, cos):

      B, S, D = x.shape
      ori_type = x.dtype
      sin = sin[:S]
      cos = cos[:S]
      x_0 = x[:, :, :D//2]
      x_1 = x[:, :, D//2:]

      y0 = x_0 * cos[None, :, :] - x_1 * sin[None, :, :]
      y1 = x_0 * sin[None, :, :] + x_1 * cos[None, :, :]
      y = torch.concat([y0, y1], dim=-1)

      return y.to(ori_type)
    roped_q = apply_rope_(q, sin, cos)
    roped_k = apply_rope_(k, sin, cos)
    return roped_q, roped_k

In [26]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [24]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
torch.Size([8, 32]) theta.max()=tensor(7.) theta.min()=tensor(0.)
  ✅ [1/4] Output shapes (3.1ms)
torch.Size([16, 16]) theta.max()=tensor(15.) theta.min()=tensor(0.)
  ✅ [2/4] Preserves norm (4.5ms)
torch.Size([8, 8]) theta.max()=tensor(7.) theta.min()=tensor(0.)
torch.Size([11, 8]) theta.max()=tensor(10.) theta.min()=tensor(0.)
  ✅ [3/4] Relative position property (3.1ms)
torch.Size([4, 4]) theta.max()=tensor(3.) theta.min()=tensor(0.)
  ✅ [4/4] Gradient flow (2.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (12.8ms total)
  Progress saved. Run status() to see your dashboard.

